# SOC-Triage-Gym — GRPO Training (Colab)

Trains a Qwen2.5-Instruct LoRA on the **Tier-1 Analyst** role using per-step GRPO with deterministic state replay. Uses `scripts/train_and_evaluate.py` as the underlying driver — this notebook is a thin Colab wrapper.

**Requirements:** GPU runtime (T4 or better). On free Colab T4, full run takes ~30 min.

**Outputs:**
- `checkpoints/soc_grpo_tier1/` — reloadable LoRA adapter
- `training_loss.png` — loss curve
- `trained_vs_baseline.png` — eval plot vs oracle
- `training_summary.json` — metrics for `demo.py`

For the **CPU-only end-to-end demo** (no training, just env walkthrough), see [`soc_triage_gym_v2.ipynb`](soc_triage_gym_v2.ipynb).

## 1. Clone the repo and install dependencies

In [ ]:
import os
REPO = 'ROHITCRAFTSYT/-Metas-OpenEnv-2'
if not os.path.exists('soc-triage-gym'):
    !git clone https://github.com/{REPO}.git soc-triage-gym
%cd soc-triage-gym
!pip install -q -e ".[dev]"
# GRPO stack — pinned for compatibility (see Pin transformers<5 in repo history)
!pip install -q "transformers<5" "trl>=0.11,<0.25" peft accelerate datasets "unsloth" "unsloth_zoo"

## 2. Restart kernel after Unsloth install

Unsloth patches the runtime at install time, so a kernel restart is required before the first training-related import. **After restarting, re-run the `%cd soc-triage-gym` line below**, then continue from cell 3.

In [ ]:
import os
if not os.path.basename(os.getcwd()).startswith('soc-triage-gym'):
    %cd soc-triage-gym
import torch
assert torch.cuda.is_available(), 'GPU runtime required. Switch runtime to T4/A100/L4.'
print('GPU:', torch.cuda.get_device_name(0))
print('bf16 supported:', torch.cuda.is_bf16_supported())

## 3. Start the SOC-Triage-Gym server in the background

GRPO training drives the env via HTTP — same wire format the judges hit on Hugging Face.

In [ ]:
import subprocess, time, httpx

server = subprocess.Popen(
    ['uvicorn', 'server.app:app', '--host', '127.0.0.1', '--port', '7860'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
for _ in range(40):
    try:
        if httpx.get('http://127.0.0.1:7860/health', timeout=2).status_code == 200:
            print('server up'); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('server failed to start within 40s')

## 4. Run the training driver

`scripts/train_and_evaluate.py` is the canonical driver — it builds the per-step GRPO dataset, trains the LoRA, evaluates against the oracle, and emits the artifacts judges look for. See `train_grpo.py` for the GRPO config knobs.

**Tunables** (set via env var to fit free Colab budgets):
- `SOC_TRAIN_TASKS` — comma-separated task IDs (default: `team_phishing_escalation,team_lateral_team`)
- `SOC_TRAIN_N_SEEDS` — number of seeds per task (default: 20)
- `NUM_EPOCHS` — GRPO epochs (default: 3)
- `NUM_GENERATIONS` — completions per prompt (default: 8 — drop to 4 on T4)

In [ ]:
import os
# Free-T4-friendly defaults — drop generations + seeds to fit 16 GB VRAM
os.environ.setdefault('NUM_GENERATIONS', '4')
os.environ.setdefault('SOC_TRAIN_N_SEEDS', '12')
os.environ.setdefault('NUM_EPOCHS', '2')

!python scripts/train_and_evaluate.py

## 5. Review the artifacts

All outputs land in the repo root. The same files are committed back to the Hugging Face Space so judges can see them without rerunning training.

In [ ]:
import json, os
from IPython.display import Image, display, Markdown

summary_path = 'training_summary.json'
if os.path.exists(summary_path):
    summary = json.load(open(summary_path))
    display(Markdown(f"""### Training Summary

| Metric | Value |
|---|---|
| Role | `{summary.get('role')}` |
| Model | `{summary.get('model_name')}` |
| Oracle avg score | {summary.get('oracle_avg', 0):.3f} |
| Trained avg score | {summary.get('trained_avg', 0):.3f} |
| Delta | {summary.get('delta', 0):+.3f} |
| Episodes | {summary.get('n_episodes', 0)} |
"""))

for img in ('training_loss.png', 'trained_vs_baseline.png', 'reward_curve_tier1_oracle.png'):
    if os.path.exists(img):
        print(f'\n— {img} —')
        display(Image(img))

## 6. Save the LoRA adapter to Hugging Face Hub (optional)

Push the trained adapter to a model repo so others can reload it via `peft.PeftModel.from_pretrained(...)`.

In [ ]:
# Uncomment and set your HF token + repo to push.
# from huggingface_hub import login, HfApi
# login(token='hf_...')
# HfApi().upload_folder(folder_path='checkpoints/soc_grpo_tier1',
#                       repo_id='YOUR_USER/soc-grpo-tier1',
#                       repo_type='model')

## Notes

- This notebook trains **only the Tier-1 role**; Tier-2 and Manager are run by the scripted oracle during evaluation. Re-run with `SOC_TRAIN_ROLE=tier2` (and a fresh checkpoint) for staged training across all three roles.
- A 0.5B–1.5B model on free Colab is unlikely to beat the strong heuristic oracle on absolute score. The contribution of this notebook is **demonstrating that the GRPO pipeline works end-to-end** and produces deterministic per-step learning signal — judges score the environment, not the trained model.
- For the read-only demo (no GPU), open [`soc_triage_gym_v2.ipynb`](soc_triage_gym_v2.ipynb).
- For the one-command judge demo, run `python demo.py` from the repo root.